# Homodyne spectrum of two beads at θhom = 0 (archived)

The homodyne spectrum $S_{hom}(\omega)$ of a 2×2 system (two beads, x and y each, coupling
$g/2\pi = -165$ kHz, $\Delta/2\pi = -450$ kHz, $\kappa/2\pi = 400$ kHz) at homodyne angle
$\theta_{hom} = 0$, and a list of its local minima in dB. The deepest minimum (≈ −3.18 dB at
±597 kHz) is the squeezing of the output light. Runs in a few seconds.

**Updated Nov 2026** to current levisim, reproducing the stored output of the original run
(minimum $S_{hom}$ = 0.4807392, all listed minima equal to the printed precision):

- `calculate_homodyne_spectrum` → `homodyne_spectrum`; the toy parameter `'φ2'` → `'φ_even_n'`; `setup_hybridisation_power_series` → `load_in_power_series`;
- the old `'pairs'` keys `first_Ωx` and `Ω_gap` are written as `Ω_first`, `Ω_interpair_gap` and
  `Ω_intrapair_gap = Ω_gap/10`, which gives the same frequencies (Ωx = 312, 297 kHz; Ωy = 313.5, 298.5 kHz);
- the run used a levisim version whose toy defaults were $g_z = 0.5$ kHz and
  $g_{xy} = g_{xz} = g_{yz} = -0.1$ kHz (all 0 today); they are set explicitly so the numbers match.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import argrelextrema
from levisim import *

set_plt_rcParams_to_xmgrace_style()
plt.rcParams['axes.xmargin'] = 0

g = -165*kHz
toy_params = {
    # the pre-2025 'pairs' keys first_Ωx = 312 kHz and Ω_gap = 15 kHz, in today's names
    'Ω_model': 'pairs',
    'Ω_first': 312*kHz,
    'Ω_interpair_gap': 15*kHz,
    'Ω_intrapair_gap': 1.5*kHz,
    'Ωz_to_Ωx_multiplier': 10,

    'toy_gx': g,
    'toy_gy': g,
    # the toy-model defaults of the levisim version that produced the stored output (all 0 now)
    'toy_gz': 0.5*kHz,
    'toy_gxy': -0.1*kHz,
    'toy_gxz': -0.1*kHz,
    'toy_gyz': -0.1*kHz,

    # phase of the even-numbered tweezers (called 'φ2' before Dec 2025)
    'φ_even_n': 0*degrees,

    'use_Γ_formula': True,
}
cavity_kwargs = {
    'Δ': -450*kHz,
    'κ': 400*kHz,
    # 'θhom': π/4,
    'θhom': 0,
}
gas_kwargs = {
    'P': 1e-8*mbar,
    'T': 300*K,
    'vth': 500,  # m/s
}
N, axes = 2, 2
ω_range = np.arange(-800, 800, 0.5) * kHz
mtr = MultiTrap(
    num_traps=N,
    Cartesian_axes_per_trap=axes,
    ω_range=ω_range,

    cavity_kwargs=cavity_kwargs,
    gas_kwargs=gas_kwargs,
    toy_params=toy_params,

    load_in_power_series=False,
)
mtr.info
mtr.info_per_trap

method = QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION

labels = [
    '$S_{x_1 x_1}$',
    '$S_{y_1 y_1}$',
    '$S_{z_1 z_1}$',
    '$S_{x_2 x_2}$',
    '$S_{y_2 y_2}$',
    '$S_{z_2 z_2}$',
]

# plt.plot(ω_range / kHz, mtr.calculate_all_hybridised_position_spectra(method), label=labels)
hom = mtr.homodyne_spectrum(method)
fig, ax = plt.subplots()
ax.plot(ω_range / kHz, hom, label='$S_{hom}$')

print(f'{min(hom)=}')
print(f'min hom is {10 * np.log10(min(hom))} decibels')

minima_ii = argrelextrema(hom, np.less)[0]

minima = hom[minima_ii]

for ii, minimum in zip(minima_ii, minima):
    print(
        f'{ω_range[ii] / kHz:.1f} kHz >> '
        f'{minimum:.3f} >> '
        f'{10 * np.log10(minimum):.3f}dB'
    )

ax.set_xlabel('ω/2π (kHz)')
ax.set_ylabel('$S_{hom}$ (shot-noise units)')
ax.grid(alpha=0.3)
ax.legend()
plt.show()